# Preprocessing & Exploratory Data Analysis (EDA)

This notebook focuses on manually extracting data from our DB, testing our preprocessing functions (like standardizing dates and filling missing values), and evaluating our Prophet AI model's accuracy on CS2 Skin datasets.

In [ ]:
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Ensure our backend modules can be imported within the notebook
sys.path.append(os.path.abspath(os.path.join('', '..')))
from app.core.database.supabase import supabase
from app.ml.data_prep import prepare_prophet_dataframe
from app.ml.predictor import CS2Predictor

print("Backend libraries successfully loaded!")

## 1. Raw Data Extraction
Let's fetch the actual stored data from Supabase for a skin to see what it looks like before preprocessing.

In [ ]:
# Fetch a mock/real item from the DB
item_res = supabase.table("items").select("*").limit(1).execute()

if not item_res.data:
    print("No items in DB. Run `uv run python test_ml_pipeline.py` first to generate mock data.")
else:
    item = item_res.data[0]
    item_id = item['id']
    hash_name = item['hash_name']
    print(f"Target Item: {hash_name}")
    
    raw_prices = supabase.table("historical_prices").select("*").eq("item_id", item_id).execute().data
    raw_players = supabase.table("player_counts").select("*").execute().data
    raw_news = supabase.table("game_updates").select("*").execute().data
    
    df_raw = pd.DataFrame(raw_prices)
    display(df_raw.head(3))

## 2. Preprocessing & Feature Engineering
We need to transform the raw tables into the unified structure Facebook Prophet expects:
`ds` (Date), `y` (Target Price), `player_count` (Regressor 1), and `is_market_shocker` (Regressor 2).

In [ ]:
try:
    df_train, df_future = prepare_prophet_dataframe(raw_prices, raw_players, raw_news)
    
    print("--- Extracted Training DataFrame ---")
    display(df_train.head())
    
    print("\n--- Generated Future Extrapolation DataFrame (Next 7 Days) ---")
    display(df_future.head(7))
    
except Exception as e:
    print(f"Preprocessing failed: {e}")

## 3. Data Visualization (EDA)
Before we train the AI, let's visualize how the price correlates with the player count and market shocks over time.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Price Line (Primary Y axis)
fig.add_trace(
    go.Scatter(x=df_train['ds'], y=df_train['y'], name="Price (USD)", mode='lines+markers'),
    secondary_y=False,
)

# Player Count Line (Secondary Y axis)
fig.add_trace(
    go.Scatter(x=df_train['ds'], y=df_train['player_count'], name="Concurrent Players", mode='lines', line=dict(dash='dot', color='rgba(255, 165, 0, 0.5)')),
    secondary_y=True,
)

# Mark Shockers (News Updates)
shockers = df_train[df_train['is_market_shocker'] == 1]
if not shockers.empty:
    fig.add_trace(
        go.Scatter(x=shockers['ds'], y=shockers['y'], name="Market Shocker", mode='markers', marker=dict(color='red', size=12, symbol='x')),
        secondary_y=False,
    )

fig.update_layout(
    title_text=f"EDA: Price vs Player Count for {hash_name}",
    hovermode="x unified"
)

fig.update_yaxes(title_text="Price (USD)", secondary_y=False)
fig.update_yaxes(title_text="Concurrent Players", secondary_y=True)

fig.show()

## 4. Prophet Model Training & Backtesting
Now we pass our preprocessed `df_train` to Prophet, generate the forecast, and plot the actual vs predicted trajectory.

In [ ]:
predictor = CS2Predictor()

# Train Prophet & Predict the future 7 days
df_forecast = predictor.train_and_forecast(df_train, df_future)

# Visualization: Past + Future Trajectory
fig_pred = go.Figure()

# Actual Prices
fig_pred.add_trace(go.Scatter(x=df_train['ds'], y=df_train['y'], name="Historical Actuals", line=dict(color='blue')))

# Predicted Future Prices
fig_pred.add_trace(go.Scatter(x=df_forecast['target_date'], y=df_forecast['predicted_price'], name="Prophet 7-Day Forecast", line=dict(color='green', dash='dash')))

fig_pred.update_layout(title=f"Prophet Forecast Projection: {hash_name}", xaxis_title="Date", yaxis_title="Price ($USD)")
fig_pred.show()